In [1]:
# STEP 0 — 建立分類器資料集（ROI crops，不分左右，統一方向）
from PIL import Image
import pandas as pd
from pathlib import Path

PROJ = Path(r"D:/中興大學/碩一上/仁愛醫院/仁愛醫院調整8總整理").resolve()  # ✅ 新專案根目錄
YOLO_ROOT = PROJ / "yolo_dataset_process"
IMG_ALL = YOLO_ROOT / "yolo_dataset" / "images"

df = pd.read_csv(PROJ/"roi_all.csv")  # 建議從 調整5 複製過來，或在6重跑一次生成

OUT = PROJ / "stage_cls_dataset"
OUT.mkdir(exist_ok=True, parents=True)
OUT_ROOT = Path(PROJ) / "outputs_bin(1),(2,3,4)"
# 統一成哪一邊的樣子？"L" = 全部變成左邊樣子；"R" = 全部變成右邊樣子
UNIFY_SIDE = "L"   # 老師說「都只有在左邊或右邊」，你可以先選 L

for _, r in df.iterrows():
    if pd.isna(r["x1"]):
        continue  # 沒偵測到跳過
    
    stage = int(r["stage"])
    side  = str(r["side"]).upper()  # 'L' 或 'R'
    
    out_dir = OUT / f"stage_{stage}"
    out_dir.mkdir(exist_ok=True, parents=True)
    
    img_path = IMG_ALL / r["filename"]
    x1, y1, x2, y2 = map(int, [r.x1, r.y1, r.x2, r.y2])
    
    with Image.open(img_path) as im:
        crop = im.crop((x1, y1, x2, y2))
        
        # 🔁 關鍵：把另一側翻成統一方向
        if UNIFY_SIDE == "L":
            # 希望所有關節都長得像「左側」 → 把右邊關節水平翻轉
            if side == "R":
                crop = crop.transpose(Image.FLIP_LEFT_RIGHT)
        elif UNIFY_SIDE == "R":
            # 希望都長得像「右側」 → 把左邊翻轉
            if side == "L":
                crop = crop.transpose(Image.FLIP_LEFT_RIGHT)
        else:
            raise ValueError("UNIFY_SIDE 只能是 'L' 或 'R'")
        
        crop.save(out_dir / r["filename"])

print("完成建立 stage_cls_dataset（已統一方向，不分左右）：", OUT)



完成建立 stage_cls_dataset（已統一方向，不分左右）： D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\stage_cls_dataset


In [2]:
import torch
import numpy as np
import random
import os

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    # 如果用 GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # 讓 cuDNN 的結果固定（禁用一些非確定性加速）
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print("Seed fixed!")


Seed fixed!


In [3]:
from torch.utils.data import Dataset

class BinaryStageWrapper(Dataset):
    def __init__(self, base_dataset):
        self.base = base_dataset
    def __len__(self):
        return len(self.base)
    def __getitem__(self, idx):
        x, y = self.base[idx]      # y: 0..3
        y2 = 0 if y == 0 else 1    # 0:stage1, 1:stage2/3/4
        return x, y2

In [4]:
# STEP 1 — 載入資料集 + train/val split
import torch
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset

DATA_ROOT = OUT

# 訓練用：有增強，但❗不要再左右翻
train_tf = T.Compose([
    T.Resize((384, 384)),
    # T.RandomHorizontalFlip(),   # 先關掉，不再打亂左右
    T.RandomRotation(10),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)
])

# 驗證用：只做必要的 resize + normalize
val_tf = T.Compose([
    T.Resize((384, 384)),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)
])

# 先只讀檔名（不指定 transform），拿到總數量
base_dataset = ImageFolder(DATA_ROOT)

print("總資料量:", len(base_dataset))
print("分類對照：", base_dataset.class_to_idx)


總資料量: 287
分類對照： {'stage_1': 0, 'stage_2': 1, 'stage_3': 2, 'stage_4': 3}


In [5]:
from pathlib import Path
import torch
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset

DATA_ROOT = OUT  # 你的 stage_cls_dataset
base_dataset = ImageFolder(DATA_ROOT)  # 只用來拿 labels，不套 transform

# ===== 你想要的比例 =====
train_ratio = 0.7
val_ratio   = 0.15
test_ratio  = 0.15
assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-8

# 固定 random seed
g = torch.Generator().manual_seed(42)

# 取得每張圖的 label
labels4 = torch.tensor([y for _, y in base_dataset.samples])  # 0..3
labels2 = (labels4 != 0).long()  # 0:(stage1) 1:(stage2,3,4)

train_idx_all, val_idx_all, test_idx_all = [], [], []
num_classes = 2

for c in range(num_classes):
    idx_c = torch.where(labels2 == c)[0]
    idx_c = idx_c[torch.randperm(len(idx_c), generator=g)]

    n_c = len(idx_c)
    n_train_c = int(n_c * train_ratio)
    n_val_c   = int(n_c * val_ratio)
    n_test_c  = n_c - n_train_c - n_val_c

    train_idx_all.append(idx_c[:n_train_c])
    val_idx_all.append(idx_c[n_train_c:n_train_c + n_val_c])
    test_idx_all.append(idx_c[n_train_c + n_val_c:])

train_idx = torch.cat(train_idx_all)
val_idx   = torch.cat(val_idx_all)
test_idx  = torch.cat(test_idx_all)

# 再打散一次（可選，但我通常會做）
train_idx = train_idx[torch.randperm(len(train_idx), generator=g)]
val_idx   = val_idx[torch.randperm(len(val_idx), generator=g)]
test_idx  = test_idx[torch.randperm(len(test_idx), generator=g)]

print("train/val/test =", len(train_idx), len(val_idx), len(test_idx))

# ===== 各自有 transform 的 dataset =====
train_dataset4 = ImageFolder(DATA_ROOT, transform=train_tf)
val_dataset4   = ImageFolder(DATA_ROOT, transform=val_tf)
test_dataset4  = ImageFolder(DATA_ROOT, transform=val_tf)

train_dataset = BinaryStageWrapper(train_dataset4)
val_dataset   = BinaryStageWrapper(val_dataset4)
test_dataset  = BinaryStageWrapper(test_dataset4)


train_set = Subset(train_dataset, train_idx.tolist())
val_set   = Subset(val_dataset,   val_idx.tolist())
test_set  = Subset(test_dataset,  test_idx.tolist())

train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=16, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=16, shuffle=False)

# 額外：看每個 split 的 class 分布（強烈建議）
def count_per_class(idxs):
    c = torch.bincount(labels2[idxs], minlength=2)
    return c.tolist()

print("per-class train:", count_per_class(train_idx))
print("per-class val  :", count_per_class(val_idx))
print("per-class test :", count_per_class(test_idx))



train/val/test = 200 42 45
per-class train: [34, 166]
per-class val  : [7, 35]
per-class test : [8, 37]


In [6]:
# ============================================================
# MUST HAVE: create_model() (same as training)
# Put this cell BEFORE you create/load m1/m2/m3
# ============================================================

import torch
import torch.nn as nn
import torchvision.models as models

def create_model(model_name: str, num_classes: int = 2):
    """
    Build a torchvision model backbone with pretrained weights,
    and replace the final classification layer to num_classes.

    model_name options (match your training notebooks):
      efficientnet_b0, efficientnet_b1, resnet50,
      convnext_tiny, convnext_small, convnext_base,
      vit_b_16, densenet121, densenet169
    """
    model_name = model_name.lower()

    if model_name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT
        model = models.efficientnet_b0(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)

    elif model_name == "efficientnet_b1":
        weights = models.EfficientNet_B1_Weights.DEFAULT
        model = models.efficientnet_b1(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)

    elif model_name == "resnet50":
        weights = models.ResNet50_Weights.DEFAULT
        model = models.resnet50(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)

    elif model_name == "convnext_tiny":
        weights = models.ConvNeXt_Tiny_Weights.DEFAULT
        model = models.convnext_tiny(weights=weights)
        in_features = model.classifier[2].in_features
        model.classifier[2] = nn.Linear(in_features, num_classes)

    elif model_name == "convnext_small":
        weights = models.ConvNeXt_Small_Weights.DEFAULT
        model = models.convnext_small(weights=weights)
        in_features = model.classifier[2].in_features
        model.classifier[2] = nn.Linear(in_features, num_classes)

    elif model_name == "convnext_base":
        weights = models.ConvNeXt_Base_Weights.DEFAULT
        model = models.convnext_base(weights=weights)
        in_features = model.classifier[2].in_features
        model.classifier[2] = nn.Linear(in_features, num_classes)

    elif model_name == "vit_b_16":
        weights = models.ViT_B_16_Weights.DEFAULT
        model = models.vit_b_16(weights=weights)
        in_features = model.heads.head.in_features
        model.heads.head = nn.Linear(in_features, num_classes)

    elif model_name == "densenet121":
        weights = models.DenseNet121_Weights.DEFAULT
        model = models.densenet121(weights=weights)
        in_features = model.classifier.in_features
        model.classifier = nn.Linear(in_features, num_classes)

    elif model_name == "densenet169":
        weights = models.DenseNet169_Weights.DEFAULT
        model = models.densenet169(weights=weights)
        in_features = model.classifier.in_features
        model.classifier = nn.Linear(in_features, num_classes)

    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    return model


In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
PROJ = Path(r"D:/中興大學/碩一上/仁愛醫院/仁愛醫院調整8總整理").resolve()

M2_TOP3 = [
    ("efficientnet_b0", PROJ / "outputs_bin(1),(2,3,4)/_best_model/best_efficientnet_b0.pth"),
    ("resnet50",        PROJ / "outputs_bin(1),(2,3,4)/resnet50/best_resnet50.pth"),
    ("convnext_tiny",     PROJ / "outputs_bin(1),(2,3,4)/convnext_tiny/best_convnext_tiny.pth"),
]


In [8]:
def load_models(model_list):
    models = []
    for name, ckpt in model_list:
        m = create_model(name, num_classes=2).to(device)
        m.load_state_dict(torch.load(ckpt, map_location=device))
        m.eval()
        models.append(m)
    return models

m2_models = load_models(M2_TOP3)


In [9]:
@torch.no_grad()
def extract_stack_features(loader, models):
    X, y = [], []

    for imgs, labels in loader:
        imgs = imgs.to(device)

        feats = []
        for m in models:
            logits = m(imgs)              # (B,2)
            probs  = F.softmax(logits, 1) # (B,2)
            feats.append(probs)

        # concat => (B, num_models*2)
        feats = torch.cat(feats, dim=1)
        X.append(feats.cpu())
        y.append(labels)

    return torch.cat(X).numpy(), torch.cat(y).numpy()


In [10]:
# 03.25_stage_classifier_m2_stacking.ipynb

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score


In [11]:
X_val, y_val = extract_stack_features(val_loader, m2_models)

meta_clf = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)
meta_clf.fit(X_val, y_val)

print("meta val acc :", accuracy_score(y_val, meta_clf.predict(X_val)))
print("meta val f1  :", f1_score(y_val, meta_clf.predict(X_val), average="macro"))


meta val acc : 0.9523809523809523
meta val f1  : 0.9142857142857143


In [12]:
class M2Stacker:
    def __init__(self, models, meta_clf, device="cuda"):
        self.models = models
        self.meta = meta_clf
        self.device = device

    @torch.no_grad()
    def p_stage1(self, x):
        x = x.to(self.device)

        feats = []
        for m in self.models:
            p = F.softmax(m(x), dim=1)  # (B,2)
            feats.append(p)

        feats = torch.cat(feats, dim=1).cpu().numpy()
        prob_stage1 = self.meta.predict_proba(feats)[:, 0]
        return torch.tensor(prob_stage1)


In [13]:
from pathlib import Path

COMB_DIR = PROJ / "03.5_combination" / "03.25_m2_stacking_top3"
(COMB_DIR / "m2_base_ckpts").mkdir(parents=True, exist_ok=True)
(COMB_DIR / "meta").mkdir(parents=True, exist_ok=True)

print("✅ COMB_DIR =", COMB_DIR)


✅ COMB_DIR = D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\03.5_combination\03.25_m2_stacking_top3


In [14]:
import shutil

for name, ckpt in M2_TOP3:
    ckpt = Path(ckpt)
    if not ckpt.exists():
        raise FileNotFoundError(f"ckpt not found: {ckpt}")

    dst = COMB_DIR / "m2_base_ckpts" / f"{name}__{ckpt.name}"
    shutil.copy2(ckpt, dst)
    print("copied:", dst)


copied: D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\03.5_combination\03.25_m2_stacking_top3\m2_base_ckpts\efficientnet_b0__best_efficientnet_b0.pth
copied: D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\03.5_combination\03.25_m2_stacking_top3\m2_base_ckpts\resnet50__best_resnet50.pth
copied: D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\03.5_combination\03.25_m2_stacking_top3\m2_base_ckpts\convnext_tiny__best_convnext_tiny.pth


In [15]:
import joblib

META_PATH = COMB_DIR / "meta" / "m2_meta_logreg.pkl"
joblib.dump(meta_clf, META_PATH)
print("✅ saved meta_clf ->", META_PATH)


✅ saved meta_clf -> D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\03.5_combination\03.25_m2_stacking_top3\meta\m2_meta_logreg.pkl


In [16]:
import json

config = {
    "task": "m2 stacking (stage1 vs stage2/3)",
    "m2_base_models": [
        {"model_name": name,
         "ckpt_in_comb_dir": str((COMB_DIR / "m2_base_ckpts" / f"{name}__{Path(ckpt).name}").resolve())}
        for name, ckpt in M2_TOP3
    ],
    "meta_clf_path": str(META_PATH.resolve()),
    "thr2_default": float(THR2) if "THR2" in globals() else 0.5,
    "seed": 42
}

CFG_PATH = COMB_DIR / "config.json"
CFG_PATH.write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")
print("✅ saved config ->", CFG_PATH)


✅ saved config -> D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\03.5_combination\03.25_m2_stacking_top3\config.json
